## Multi User Personal Model Evaluation 

Notebook evaluates shortlisted personal models against global/base model.
This is done with the aim of finding the optimal blending thresehold (i.e. num of runs until global model blends with personal and num of runs before system should change to fully personal model).

In [ ]:
from pathlib import Path

# constants
DB_PATH = "/Users/darraghdonnelly/dev/Database/personalised_eval_db.db"
BASE_MODEL_PATH = Path("/Users/darraghdonnelly/dev/fyp-running-app/backend/ml_models/base_model.joblib")
MIN_HISTORY = 1

# user info for global model
USER_INFO = {
    1: {"age": 22, "sex": 0},
    2: {"age": 24, "sex": 0},
}


In [437]:
import sqlite3
import pandas as pd

with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql_query(
        "SELECT user_id, distance, time, weather_temp, weather_precip_mm, weather_humidity, weather_wind_kph, completed_at FROM concat_users",
        conn,
    )

# convert dist and time to km and mins
df["distance_km"] = df["distance"].astype(float) / 1000.0
df["time_min"] = df["time"].astype(float) / 60.0

# set up feature cols  
personal_feature_cols = [
    "distance_km",
    "weather_temp",
    "weather_precip_mm",
    "weather_humidity",
    "weather_wind_kph",
]
# num of runs each users' recorded and df layout
print(df.groupby("user_id").size())
df.head()


user_id
1    20
2    91
dtype: int64


,user_id,distance,time,weather_temp,weather_precip_mm,weather_humidity,weather_wind_kph,completed_at,distance_km,time_min
0,1,5000.0,1373,14.0,0.9,56.0,14.04,2024-07-18T08:34:00Z,5.00,22.883333
1,1,6760.0,2260,12.0,0.0,83.0,17.64,2024-09-10T20:05:00Z,6.76,37.666667
2,1,5070.0,1383,11.0,0.0,85.0,9.36,2024-09-24T13:41:00Z,5.07,23.050000
3,1,5020.0,1599,11.0,15.0,77.0,35.64,2025-02-03T11:09:00Z,5.02,26.650000
4,1,7170.0,2479,4.0,0.0,75.0,12.24,2025-02-07T18:27:00Z,7.17,41.316667


In [ ]:
import joblib
import numpy as np
import pandas as pd

# global model prediciton
def predict_global_seconds(base_artifact, distance_m, age=None, sex=None):

    # check model exists
    if base_artifact is None:
        return None

    # unpack model and features 
    model = base_artifact.get("model")
    feature_order = base_artifact.get("feature_order")

    feats = {
        "log_distance": np.log(float(distance_m)),
        "age": float(age),
        "sex": float(sex)
    }

    # feature vector stores feats in expected oreder for model
    vector = [[feats[name] for name in feature_order]]

    # predict for all features
    pred = float(model.predict(vector)[0])

    # global model predicts log(time)
    return float(np.exp(pred))

# group by num of runs recorded
def bucket_label(runs_recorded):
    if 1 <= runs_recorded <= 4:
        return "1-4"
    if 5 <= runs_recorded <= 8:
        return "5-8"
    if 9 <= runs_recorded <= 12:
        return "9-12"
    return None

base_model = joblib.load(BASE_MODEL_PATH)


In [446]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge

results = []

# rolling prediction - 
# loop through dataset split based on user
for user_id, user_runs in df.groupby("user_id"):

    # range of runs to evaluate 
    for num_runs_used in range(MIN_HISTORY, len(user_runs)):

        # store all earlier runs in train df
        train_df = user_runs.iloc[:num_runs_used]

        # get actual times from run records
        test_row = user_runs.iloc[num_runs_used]

        # train model on available personal featurs from train_df
        X_train = train_df[personal_feature_cols]

        # actual run times for training
        y_train = train_df["time_min"]

        # store available features for test run
        X_test = user_runs.iloc[[num_runs_used]][personal_feature_cols]

        actual_min = float(test_row["time_min"])

        # run simple baseline model 
        mean_model = DummyRegressor(strategy="mean")
        mean_model.fit(X_train, y_train)
        mean_pred = float(mean_model.predict(X_test)[0])

        # store result metrics
        results.append({
            "user_id": user_id,
            "num_runs_used": num_runs_used,
            "model": "Mean Baseline",
            "actual_min": actual_min,
            "pred_min": mean_pred,
            "abs_error_min": abs(actual_min - mean_pred),
            "squared_error_min": (actual_min - mean_pred) ** 2,
        })

        # repeat process for all models 
        
        linear_model = LinearRegression()
        linear_model.fit(X_train, y_train)
        linear_pred = float(linear_model.predict(X_test)[0])

        results.append({
            "user_id": user_id,
            "num_runs_used": num_runs_used,
            "model": "Personal Linear",
            "actual_min": actual_min,
            "pred_min": linear_pred,
            "abs_error_min": abs(actual_min - linear_pred),
            "squared_error_min": (actual_min - linear_pred) ** 2,
        })

        ridge_model = Ridge(alpha=0.1)
        ridge_model.fit(X_train, y_train)
        ridge_pred = float(ridge_model.predict(X_test)[0])

        results.append({
            "user_id": user_id,
            "num_runs_used": num_runs_used,
            "model": "Personal Ridge",
            "actual_min": actual_min,
            "pred_min": ridge_pred,
            "abs_error_min": abs(actual_min - ridge_pred),
            "squared_error_min": (actual_min - ridge_pred) ** 2,
        })

        # run global model on dist, sex and age feat only
        info = USER_INFO.get(user_id, {})
        global_pred_seconds = predict_global_seconds(
            base_model,
            float(test_row["distance"]),
            age=info.get("age"),
            sex=info.get("sex"),
        )
        global_pred_min = global_pred_seconds / 60.0
        results.append({
            "user_id": user_id,
            "num_runs_used": num_runs_used,
            "model": "Global Model",
            "actual_min": actual_min,
            "pred_min": global_pred_min,
            "abs_error_min": abs(actual_min - global_pred_min),
            "squared_error_min": (actual_min - global_pred_min) ** 2,
        })

        blend_upper_bound = 6
        blend_lower_bound = 4

        # blending logic from main.py
        if num_runs_used < blend_lower_bound:
            blended_pred_min = global_pred_min
        else:
            weight = (num_runs_used - blend_lower_bound) / float(blend_upper_bound - blend_lower_bound)
            if weight < 0:
                weight = 0.0
            if weight > 1:
                weight = 1.0
            blended_pred_min = (weight * ridge_pred) + ((1 - weight) * global_pred_min)

        results.append({
            "user_id": user_id,
            "num_runs_used": num_runs_used,
            "model": "Blended Model",
            "actual_min": actual_min,
            "pred_min": blended_pred_min,
            "abs_error_min": abs(actual_min - blended_pred_min),
            "squared_error_min": (actual_min - blended_pred_min) ** 2,
        })

predictions = pd.DataFrame(results)
predictions.head(20)


/Users/darraghdonnelly/dev/fyp-running-app/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/Users/darraghdonnelly/dev/fyp-running-app/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/Users/darraghdonnelly/dev/fyp-running-app/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/Users/darraghdonnelly/dev/fyp-running-app/.venv/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but GradientBoostingRegressor was fitted with feature names
  warnings.warn(
/Users/darraghdonnelly/dev/fyp-running-app/.

,user_id,num_runs_used,model,actual_min,pred_min,abs_error_min,squared_error_min
0,1,1,Mean Baseline,37.666667,22.883333,14.783333,218.546944
1,1,1,Personal Linear,37.666667,22.883333,14.783333,218.546944
2,1,1,Personal Ridge,37.666667,22.883333,14.783333,218.546944
3,1,1,Global Model,37.666667,31.402947,6.263720,39.234186
4,1,1,Blended Model,37.666667,31.402947,6.263720,39.234186
5,1,2,Mean Baseline,23.050000,30.275000,7.225000,52.200625
6,1,2,Personal Linear,23.050000,38.124392,15.074392,227.237299
7,1,2,Personal Ridge,23.050000,38.122299,15.072299,227.174203
8,1,2,Global Model,23.050000,31.402947,8.352947,69.771721
9,1,2,Blended Model,23.050000,31.402947,8.352947,69.771721


In [ ]:
# group overall results by model and get mean abs err and sqrd err
overall_summary = (
    predictions.groupby("model")[["abs_error_min", "squared_error_min"]]
    .mean()
    .reset_index()
)
# convert results to mae and rmse
overall_summary["mae_min"] = overall_summary["abs_error_min"]
overall_summary["rmse_min"] = overall_summary["squared_error_min"] ** 0.5

overall_summary[["model", "mae_min", "rmse_min"]]


,model,mae_min,rmse_min
0,Blended Model,6.985007,8.862387
1,Global Model,14.203125,21.842820
2,Mean Baseline,29.802666,39.944933
3,Personal Linear,8.079626,10.665211
4,Personal Ridge,7.901785,10.355200


In [ ]:
# get user by user summary of overall model results
by_user_summary = (
    predictions.groupby(["user_id", "model"])[["abs_error_min", "squared_error_min"]]
    .mean()
    .reset_index()
)

by_user_summary["mae_min"] = by_user_summary["abs_error_min"]
by_user_summary["rmse_min"] = by_user_summary["squared_error_min"] ** 0.5
by_user_summary = by_user_summary.sort_values(["user_id", "mae_min"])

by_user_summary[["user_id", "model", "mae_min", "rmse_min"]]


,user_id,model,mae_min,rmse_min
0,1,Blended Model,4.634197,6.063791
4,1,Personal Ridge,8.136079,12.833675
3,1,Personal Linear,8.205929,12.904832
1,1,Global Model,9.200417,10.701002
2,1,Mean Baseline,10.383735,13.731870
5,2,Blended Model,7.481289,9.346685
9,2,Personal Ridge,7.852323,9.751768
8,2,Personal Linear,8.052962,10.129295
6,2,Global Model,15.259252,23.529920
7,2,Mean Baseline,33.902218,43.504460


In [ ]:
# summary of results based on number of runs available
history_summary = (
    predictions.groupby(["num_runs_used", "model"])[["abs_error_min", "squared_error_min"]]
    .mean()
    .reset_index()
)

history_summary["mae_min"] = history_summary["abs_error_min"]
history_summary["rmse_min"] = history_summary["squared_error_min"] ** 0.5

history_summary[["num_runs_used", "model", "mae_min", "rmse_min"]]

,num_runs_used,model,mae_min,rmse_min
0,1,Blended Model,5.216177,5.320324
1,1,Global Model,5.216177,5.320324
2,1,Mean Baseline,17.616667,17.843058
3,1,Personal Linear,17.616667,17.843058
4,1,Personal Ridge,17.616667,17.843058
...,...,...,...,...
445,90,Blended Model,12.499407,12.499407
446,90,Global Model,17.994276,17.994276
447,90,Mean Baseline,9.463704,9.463704
448,90,Personal Linear,12.499128,12.499128


In [ ]:
# results sorted by num runs models were trained on

# copy previous predictions 
bucketed_predictions = predictions.copy()

# apply bucket labels to each row
bucketed_predictions["history_bucket"] = bucketed_predictions["num_runs_used"].apply(bucket_label)

# get abs and mse for each bucket and model 
bucket_summary = (
    bucketed_predictions.groupby(["history_bucket", "model"])[["abs_error_min", "squared_error_min"]]
    .mean()
    .reset_index()
)

bucket_summary["mae_min"] = bucket_summary["abs_error_min"]
bucket_summary["rmse_min"] = bucket_summary["squared_error_min"] ** 0.5

# pivot mae results with buckets by col and model by row
bucket_pivot = bucket_summary.pivot(index="model", columns="history_bucket", values="mae_min")
bucket_pivot

history_bucket,1-4,5-8,9-12
model,,,
Blended Model,11.892651,5.983560,6.272992
Global Model,11.892651,9.891057,7.412969
Mean Baseline,15.397396,18.049640,19.997600
Personal Linear,22.532118,8.063983,6.407289
Personal Ridge,22.495570,7.871747,6.272992
